# Qwen3-ASR Speech Recognition with OpenVINO™

The Qwen3-ASR family includes Qwen3-ASR-1.7B and Qwen3-ASR-0.6B, which support language identification and ASR for 52 languages and dialects. Both leverage large-scale speech training data and the strong audio understanding capability of their foundation model, Qwen3-Omni.

* **All-in-one**: language identification and speech recognition for many languages, including English accents from multiple countries and regions.
* **Excellent and Fast**: high-quality, robust recognition; the 0.6B version offers an accuracy-efficiency trade-off and supports long audio.
* **Forced alignment**: `Qwen3-ForcedAligner-0.6B` predicts word-level timestamps in 11 languages.

In this tutorial we run **Qwen3-ASR** with [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) via `openvino_genai.ASRPipeline`, and predict word-level timestamps with the **Qwen3-ForcedAligner** using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index). The models are exported to OpenVINO IR with Optimum Intel.

More details: original [repository](https://github.com/QwenLM/Qwen3-ASR) and [model card](https://huggingface.co/Qwen/Qwen3-ASR-0.6B-hf).

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model to OpenVINO IR](#Convert-model-to-OpenVINO-IR)
- [Select inference device](#Select-inference-device)
- [Run speech recognition](#Run-speech-recognition)
- [Word-level timestamps with the forced aligner](#Word-level-timestamps-with-the-forced-aligner)
- [Interactive demo](#Interactive-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-asr/qwen3-asr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [ ]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")

### Install dependencies

Speech recognition runs through **OpenVINO GenAI** (`openvino_genai.ASRPipeline`), so the OpenVINO C++ runtime performs generation and decoding. `transformers` / `optimum-intel` are needed only to **export** the models to OpenVINO IR and to run the optional Qwen3-ForcedAligner.

`transformers` is pinned to **`>=4.45,<5.14`** (the range required by the `optimum-intel` build above): newer builds changed `GenerationMixin.generate` (it now calls `get_experts_implementation`, which the `optimum-intel` OpenVINO model wrapper does not implement) and are declared incompatible with the `optimum-intel` build used here. OpenVINO GenAI support for Qwen3-ASR currently ships in the **nightly** wheels, so we install `openvino` / `openvino-tokenizers` / `openvino-genai` from the pre-release channel.


In [ ]:
from pip_helper import pip_install

# 1. Base runtime dependencies
pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.6",
    "torchaudio",
    "librosa",
    "soundfile",
    "gradio>=4.19",
    "scipy",
    "nncf>=2.19.0",
)

# 2. OpenVINO + OpenVINO GenAI. Qwen3-ASR support in ASRPipeline is only available
#    in the pre-release (nightly) wheels for now.
pip_install(
    "-q",
    "--pre",
    "-U",
    "openvino",
    "openvino-tokenizers",
    "openvino-genai",
    "--extra-index-url",
    "https://storage.openvinotoolkit.org/simple/wheels/nightly",
)

# 3. Optimum Intel with HuggingFace-native Qwen3-ASR / Qwen3-ForcedAligner support.
#    Used only to export the models to OpenVINO IR and to run the forced aligner.
pip_install("-q", "git+https://github.com/openvino-dev-samples/optimum-intel.git@add-qwen3-asr-hf-and-forced-aligner")

# 4. transformers is pinned to match the optimum-intel build above (its setup.py
#    requires `transformers>=4.45,<5.14`). Newer builds changed GenerationMixin.generate
#    (it now requires `get_experts_implementation`, which the OpenVINO model wrapper
#    does not implement) and are incompatible with that optimum-intel build.
#    Installed last so it is not upgraded by the optimum-intel dependency resolution.
pip_install("-q", "transformers>=4.45,<5.14", "safetensors>=0.8.0")

## Select model
[back to top ⬆️](#Table-of-contents:)

Select the Qwen3-ASR variant. The 0.6B model is recommended for faster inference.


In [ ]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen3-ASR-0.6B-hf",
    "Qwen/Qwen3-ASR-1.7B-hf",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_selector

## Convert model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Optimum Intel exposes the same model API as 🤗 Transformers, with an OpenVINO backend. `OVModelForSpeechSeq2Seq` exports the model to OpenVINO IR on the fly (`export=True`) and splits it into an audio encoder and a text decoder. Together with the model, the export also produces the OpenVINO tokenizer/detokenizer that `openvino_genai.ASRPipeline` uses at inference time.

This is equivalent to the CLI export:

```bash
optimum-cli export openvino --model Qwen/Qwen3-ASR-0.6B-hf --task automatic-speech-recognition Qwen3-ASR-0.6B-hf-ov
```

You can also compress the model weights to INT8 by passing `--weight-format int8` to the CLI, or a `quantization_config` / `load_in_8bit=True` to `from_pretrained`.


In [ ]:
from optimum.intel import OVModelForSpeechSeq2Seq
from optimum.exporters.openvino.convert import export_tokenizer
from transformers import AutoProcessor, AutoTokenizer

model_id = model_selector.value
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-ov")

# Set to True to compress the model weights to INT8
load_in_8bit = False

if not ov_model_dir.exists():
    ov_model = OVModelForSpeechSeq2Seq.from_pretrained(model_id, export=True, load_in_8bit=load_in_8bit)
    ov_model.save_pretrained(ov_model_dir)
    # Save the processor (preprocessor config) and export the OpenVINO tokenizer /
    # detokenizer -- ASRPipeline needs `openvino_tokenizer.xml` /
    # `openvino_detokenizer.xml`, which `save_pretrained` does not produce on its own.
    AutoProcessor.from_pretrained(model_id).save_pretrained(ov_model_dir)
    export_tokenizer(AutoTokenizer.from_pretrained(model_id), ov_model_dir)
    del ov_model
    print(f"Model exported to {ov_model_dir}")
else:
    print(f"Model already exported to {ov_model_dir}")

## Select inference device
[back to top ⬆️](#Table-of-contents:)


In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

## Run speech recognition
[back to top ⬆️](#Table-of-contents:)

We create an `openvino_genai.ASRPipeline` from the exported IR and run transcription. The pipeline expects a mono waveform resampled to 16 kHz (a list of floats); it takes care of the Qwen3-ASR prompt formatting, generation and decoding internally, so no `transformers` processor is needed at inference time.


In [ ]:
import openvino_genai as ov_genai

asr_pipe = ov_genai.ASRPipeline(str(ov_model_dir), device.value)

In [ ]:
import urllib.request
import librosa

sample_audio = Path("asr_en.wav")
if not sample_audio.exists():
    urllib.request.urlretrieve("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio)

# ASRPipeline expects a mono waveform at 16 kHz as a list of floats.
audio, sr = librosa.load(sample_audio, sr=16000)

result = asr_pipe.generate(audio.tolist())
print(f"Transcription: {result}")

By default the language is detected automatically. If it is known in advance, you can pass a language hint to skip detection — for Qwen3-ASR this is the **language name** (for Whisper models it would be a token like `"<|en|>"`):

```python
result = asr_pipe.generate(audio.tolist(), language="English")
```

> **Note.** The `task="translate"`, `return_timestamps` and `word_timestamps` arguments of `ASRPipeline` are **Whisper-specific and are ignored for Qwen3-ASR**. In particular, `ASRPipeline` does not produce word-level timestamps for Qwen3-ASR — that is what the dedicated **Qwen3-ForcedAligner** below is for.


## Word-level timestamps with the forced aligner
[back to top ⬆️](#Table-of-contents:)

`Qwen3-ForcedAligner-0.6B` predicts word-level timestamps for a known transcript. With Optimum Intel it is exposed as `OVModelForQwen3ASRForcedAligner`, used exactly like the original `Qwen3ASRForTokenClassification`: build inputs with `processor.prepare_forced_aligner_inputs`, run a single forward pass, and decode with `processor.decode_forced_alignment`.


In [ ]:
from transformers import AutoProcessor
from optimum.intel import OVModelForQwen3ASRForcedAligner

aligner_id = "Qwen/Qwen3-ForcedAligner-0.6B-hf"
aligner_dir = Path("Qwen3-ForcedAligner-0.6B-hf-ov")

if not aligner_dir.exists():
    aligner_model = OVModelForQwen3ASRForcedAligner.from_pretrained(aligner_id, export=True)
    aligner_model.save_pretrained(aligner_dir)
    del aligner_model

aligner_processor = AutoProcessor.from_pretrained(aligner_id)
aligner_model = OVModelForQwen3ASRForcedAligner.from_pretrained(aligner_dir, device=device.value)

In [ ]:
import torch

# Use the transcription produced by the GenAI ASR pipeline above.
transcript = str(result)
# The forced aligner needs the source language; set it to the audio language
# (Qwen3-ASR auto-detects it, here the sample is English).
language = "English"

aligner_inputs, word_lists = aligner_processor.prepare_forced_aligner_inputs(
    audio=audio,
    transcript=transcript,
    language=language,
)
aligner_inputs = aligner_inputs.to(aligner_model.device)

with torch.inference_mode():
    outputs = aligner_model(**aligner_inputs)

timestamps = aligner_processor.decode_forced_alignment(
    logits=outputs.logits,
    input_ids=aligner_inputs["input_ids"],
    word_lists=word_lists,
    timestamp_token_id=aligner_model.config.timestamp_token_id,
)[0]

for item in timestamps:
    print(f"{item['start_time']:6.2f}s - {item['end_time']:6.2f}s  {item['text']}")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Launch a Gradio demo to upload or record audio, transcribe it, and view word-level timestamps.


In [ ]:
from gradio_helper import make_demo

demo = make_demo(asr_pipe=asr_pipe, aligner_model=aligner_model, aligner_processor=aligner_processor)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port:
#   demo.launch(server_name='your_server_name', server_port=7860)